In [3]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        lambda_home = torch.exp(log_lambda_home).clamp(max=8.0)
        lambda_away = torch.exp(log_lambda_away).clamp(max=8.0)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.02)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    perda.backward()
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = (gols_m > gols_v).mean() * 100
    prob_e = (gols_m == gols_v).mean() * 100
    prob_v = (gols_m < gols_v).mean() * 100

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [f"{p[0][0]}x{p[0][1]} ({p[1]/n_simulacoes*100:.1f}%)" for p in placares],
    }


# exemplo de uso — ajuste os nomes pra exatamente como aparecem no CSV
resultado = prever_jogo("Algeria", "Austria")
print(resultado)

Usando device: cuda
GPU: NVIDIA GeForce RTX 3090
Total de partidas carregadas: 15889
Total de seleções no dataset: 312
ForcaSelecaoModel(
  (ataque): Embedding(312, 8)
  (defesa): Embedding(312, 8)
)
Época   0 | Perda: nan
Época  50 | Perda: nan
Época 100 | Perda: nan
Época 150 | Perda: nan
Época 200 | Perda: nan
Época 250 | Perda: nan
✅ Treinamento concluído.


ValueError: lam < 0 or lam is NaN

In [4]:
# diagnóstico rápido
print("NaNs em home_score:", df["home_score"].isna().sum())
print("NaNs em away_score:", df["away_score"].isna().sum())
print("Valores de lambda_h e lambda_a treinados:")
with torch.no_grad():
    lh, la = modelo(home_idx, away_idx)
    print("lambda_h tem NaN?", torch.isnan(lh).any().item())
    print("lambda_a tem NaN?", torch.isnan(la).any().item())
print("Pesos do modelo têm NaN?", any(torch.isnan(p).any().item() for p in modelo.parameters()))

NaNs em home_score: 18
NaNs em away_score: 18
Valores de lambda_h e lambda_a treinados:
lambda_h tem NaN? True
lambda_a tem NaN? True
Pesos do modelo têm NaN? True


In [2]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

# remove linhas com placar ausente/inválido (causa comum de NaN no treino)
df = df.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).reset_index(drop=True)
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)].reset_index(drop=True)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        log_lambda_home = log_lambda_home.clamp(min=-5.0, max=3.0)
        log_lambda_away = log_lambda_away.clamp(min=-5.0, max=3.0)

        lambda_home = torch.exp(log_lambda_home)
        lambda_away = torch.exp(log_lambda_away)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.005)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    if torch.isnan(perda):
        print(f"⚠️ Loss virou NaN na época {epoca} — pare e revise os dados/lr.")
        break

    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)  # evita explosão de gradiente
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = (gols_m > gols_v).mean() * 100
    prob_e = (gols_m == gols_v).mean() * 100
    prob_v = (gols_m < gols_v).mean() * 100

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [f"{p[0][0]}x{p[0][1]} ({p[1]/n_simulacoes*100:.1f}%)" for p in placares],
    }


# exemplo de uso — ajuste os nomes pra exatamente como aparecem no CSV
resultado = prever_jogo("Sengal", "Iraque")
print(resultado)

Usando device: cuda
GPU: NVIDIA GeForce RTX 3090
Total de partidas carregadas: 15871
Total de seleções no dataset: 312
ForcaSelecaoModel(
  (ataque): Embedding(312, 8)
  (defesa): Embedding(312, 8)
)
Época   0 | Perda: 3.7215
Época  50 | Perda: 3.3574
Época 100 | Perda: 3.0914
Época 150 | Perda: 3.0267
Época 200 | Perda: 2.9985
Época 250 | Perda: 2.9826
✅ Treinamento concluído.


ValueError: Seleção não encontrada no histórico: Sengal / Iraque

In [3]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

# remove linhas com placar ausente/inválido (causa comum de NaN no treino)
df = df.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).reset_index(drop=True)
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)].reset_index(drop=True)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        log_lambda_home = log_lambda_home.clamp(min=-5.0, max=3.0)
        log_lambda_away = log_lambda_away.clamp(min=-5.0, max=3.0)

        lambda_home = torch.exp(log_lambda_home)
        lambda_away = torch.exp(log_lambda_away)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.005)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    if torch.isnan(perda):
        print(f"⚠️ Loss virou NaN na época {epoca} — pare e revise os dados/lr.")
        break

    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)  # evita explosão de gradiente
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# ════════════════════════════════════════════════════════
# CÉLULA 7 — Rodar todos os jogos de amanhã e salvar JSON final
# Nomes no padrão do dataset martj42/international_results
# ════════════════════════════════════════════════════════
JOGOS_AMANHA = [
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Senegal",
        "visitante_nome": "Iraq",
        "contexto": "Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.",
        "grupo": "I"
    },
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Norway",
        "visitante_nome": "France",
        "contexto": "Decide liderança do Grupo I. Ambas já classificadas. Jogo de prestígio.",
        "grupo": "I"
    },
    {
        "horario": "19h (Brasília)",
        "mandante_nome": "Argentina",
        "visitante_nome": "Austria",
        "contexto": "Decide quem fecha o Grupo J em 1º. Argentina já classificada, mas Áustria ainda briga por vaga.",
        "grupo": "J"
    },
    {
        "horario": "22h (Brasília)",
        "mandante_nome": "Algeria",
        "visitante_nome": "Jordan",
        "contexto": "Argélia briga pelo 2º lugar do Grupo J. Jordânia eliminada.",
        "grupo": "J"
    },
]

resultados = []

print("=" * 60)
print("  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)")
print("  Cultura Eclética x RecomendeMe")
print("=" * 60)

for jogo in JOGOS_AMANHA:
    resultado = prever_jogo(jogo["mandante_nome"], jogo["visitante_nome"])
    resultado["horario"] = jogo["horario"]
    resultado["contexto"] = jogo["contexto"]
    resultado["grupo"] = jogo["grupo"]
    resultados.append(resultado)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {resultado['mandante']} x {resultado['visitante']}")
    print(f"   📊 Vitória {resultado['mandante']}: {resultado['prob_mandante']}%")
    print(f"   📊 Empate:                  {resultado['prob_empate']}%")
    print(f"   📊 Vitória {resultado['visitante']}: {resultado['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {resultado['palpite_maquina']}")
    print(f"   📌 Placares mais prováveis:")
    for p in resultado["placares_frequentes"]:
        print(f"      {p['placar']} → {p['freq']}% das simulações")
    print(f"   💬 {jogo['contexto']}")

with open("palpites_maquina.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "embeddings ataque/defesa treinados (PyTorch + GPU)",
        "jogos": resultados
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados salvos em palpites_maquina.json")
print("=" * 60)

Usando device: cuda
GPU: NVIDIA GeForce RTX 3090
Total de partidas carregadas: 15871
Total de seleções no dataset: 312
ForcaSelecaoModel(
  (ataque): Embedding(312, 8)
  (defesa): Embedding(312, 8)
)
Época   0 | Perda: 3.7222
Época  50 | Perda: 3.3583
Época 100 | Perda: 3.0846
Época 150 | Perda: 3.0234
Época 200 | Perda: 2.9950
Época 250 | Perda: 2.9786
✅ Treinamento concluído.
  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)
  Cultura Eclética x RecomendeMe

🕐 16h (Brasília) — Grupo I
   Senegal x Iraq
   📊 Vitória Senegal: 77.1%
   📊 Empate:                  18.2%
   📊 Vitória Iraq: 4.7%
   🤖 Palpite da Máquina: 1x0
   📌 Placares mais prováveis:
      1x0 → 20.5% das simulações
      2x0 → 20.4% das simulações
      3x0 → 13.1% das simulações
   💬 Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.

🕐 16h (Brasília) — Grupo I
   Norway x France
   📊 Vitória Norway: 5.0%
   📊 Empate:                  15.3%
   📊 Vitória France: 79.7%
   🤖 Palpi

NameError: name 'json' is not defined

In [4]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

# remove linhas com placar ausente/inválido (causa comum de NaN no treino)
df = df.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).reset_index(drop=True)
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)].reset_index(drop=True)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        log_lambda_home = log_lambda_home.clamp(min=-5.0, max=3.0)
        log_lambda_away = log_lambda_away.clamp(min=-5.0, max=3.0)

        lambda_home = torch.exp(log_lambda_home)
        lambda_away = torch.exp(log_lambda_away)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.005)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    if torch.isnan(perda):
        print(f"⚠️ Loss virou NaN na época {epoca} — pare e revise os dados/lr.")
        break

    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)  # evita explosão de gradiente
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# ════════════════════════════════════════════════════════
# CÉLULA 7 — Rodar todos os jogos de amanhã e salvar JSON final
# Nomes no padrão do dataset martj42/international_results
# ════════════════════════════════════════════════════════
JOGOS_AMANHA = [
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Senegal",
        "visitante_nome": "Iraq",
        "contexto": "Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.",
        "grupo": "I"
    },
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Norway",
        "visitante_nome": "France",
        "contexto": "Decide liderança do Grupo I. Ambas já classificadas. Jogo de prestígio.",
        "grupo": "I"
    },
    {
        "horario": "19h (Brasília)",
        "mandante_nome": "Argentina",
        "visitante_nome": "Austria",
        "contexto": "Decide quem fecha o Grupo J em 1º. Argentina já classificada, mas Áustria ainda briga por vaga.",
        "grupo": "J"
    },
    {
        "horario": "22h (Brasília)",
        "mandante_nome": "Algeria",
        "visitante_nome": "Jordan",
        "contexto": "Argélia briga pelo 2º lugar do Grupo J. Jordânia eliminada.",
        "grupo": "J"
    },
]

resultados = []

print("=" * 60)
print("  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)")
print("  Cultura Eclética x RecomendeMe")
print("=" * 60)

for jogo in JOGOS_AMANHA:
    resultado = prever_jogo(jogo["mandante_nome"], jogo["visitante_nome"])
    resultado["horario"] = jogo["horario"]
    resultado["contexto"] = jogo["contexto"]
    resultado["grupo"] = jogo["grupo"]
    resultados.append(resultado)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {resultado['mandante']} x {resultado['visitante']}")
    print(f"   📊 Vitória {resultado['mandante']}: {resultado['prob_mandante']}%")
    print(f"   📊 Empate:                  {resultado['prob_empate']}%")
    print(f"   📊 Vitória {resultado['visitante']}: {resultado['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {resultado['palpite_maquina']}")
    print(f"   📌 Placares mais prováveis:")
    for p in resultado["placares_frequentes"]:
        print(f"      {p['placar']} → {p['freq']}% das simulações")
    print(f"   💬 {jogo['contexto']}")

with open("palpites_maquina.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "embeddings ataque/defesa treinados (PyTorch + GPU)",
        "jogos": resultados
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados salvos em palpites_maquina.json")
print("=" * 60)

Usando device: cuda
GPU: NVIDIA GeForce RTX 3090
Total de partidas carregadas: 15871
Total de seleções no dataset: 312
ForcaSelecaoModel(
  (ataque): Embedding(312, 8)
  (defesa): Embedding(312, 8)
)
Época   0 | Perda: 3.7210
Época  50 | Perda: 3.3530
Época 100 | Perda: 3.0859
Época 150 | Perda: 3.0232
Época 200 | Perda: 2.9940
Época 250 | Perda: 2.9778
✅ Treinamento concluído.
  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)
  Cultura Eclética x RecomendeMe

🕐 16h (Brasília) — Grupo I
   Senegal x Iraq
   📊 Vitória Senegal: 95.9%
   📊 Empate:                  3.5%
   📊 Vitória Iraq: 0.6%
   🤖 Palpite da Máquina: 4x0
   📌 Placares mais prováveis:
      4x0 → 15.2% das simulações
      3x0 → 15.0% das simulações
      5x0 → 12.4% das simulações
   💬 Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.

🕐 16h (Brasília) — Grupo I
   Norway x France
   📊 Vitória Norway: 11.7%
   📊 Empate:                  17.6%
   📊 Vitória France: 70.7%
   🤖 Palpi

In [5]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

# remove linhas com placar ausente/inválido (causa comum de NaN no treino)
df = df.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).reset_index(drop=True)
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)].reset_index(drop=True)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        log_lambda_home = log_lambda_home.clamp(min=-5.0, max=3.0)
        log_lambda_away = log_lambda_away.clamp(min=-5.0, max=3.0)

        lambda_home = torch.exp(log_lambda_home)
        lambda_away = torch.exp(log_lambda_away)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.005)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    if torch.isnan(perda):
        print(f"⚠️ Loss virou NaN na época {epoca} — pare e revise os dados/lr.")
        break

    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)  # evita explosão de gradiente
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# ════════════════════════════════════════════════════════
# CÉLULA 7 — Rodar todos os jogos de amanhã e salvar JSON final
# Nomes no padrão do dataset martj42/international_results
# ════════════════════════════════════════════════════════
JOGOS_AMANHA = [
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Senegal",
        "visitante_nome": "Iraq",
        "contexto": "Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.",
        "grupo": "I"
    },
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Norway",
        "visitante_nome": "France",
        "contexto": "Decide liderança do Grupo I. Ambas já classificadas. Jogo de prestígio.",
        "grupo": "I"
    },
    {
        "horario": "19h (Brasília)",
        "mandante_nome": "Argentina",
        "visitante_nome": "Austria",
        "contexto": "Decide quem fecha o Grupo J em 1º. Argentina já classificada, mas Áustria ainda briga por vaga.",
        "grupo": "J"
    },
    {
        "horario": "22h (Brasília)",
        "mandante_nome": "Algeria",
        "visitante_nome": "Jordan",
        "contexto": "Argélia briga pelo 2º lugar do Grupo J. Jordânia eliminada.",
        "grupo": "J"
    },
]

resultados = []

print("=" * 60)
print("  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)")
print("  Cultura Eclética x RecomendeMe")
print("=" * 60)

for jogo in JOGOS_AMANHA:
    resultado = prever_jogo(jogo["mandante_nome"], jogo["visitante_nome"])
    resultado["horario"] = jogo["horario"]
    resultado["contexto"] = jogo["contexto"]
    resultado["grupo"] = jogo["grupo"]
    resultados.append(resultado)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {resultado['mandante']} x {resultado['visitante']}")
    print(f"   📊 Vitória {resultado['mandante']}: {resultado['prob_mandante']}%")
    print(f"   📊 Empate:                  {resultado['prob_empate']}%")
    print(f"   📊 Vitória {resultado['visitante']}: {resultado['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {resultado['palpite_maquina']}")
    print(f"   📌 Placares mais prováveis:")
    for p in resultado["placares_frequentes"]:
        print(f"      {p['placar']} → {p['freq']}% das simulações")
    print(f"   💬 {jogo['contexto']}")

with open("palpites_maquina.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "embeddings ataque/defesa treinados (PyTorch + GPU)",
        "jogos": resultados
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados salvos em palpites_maquina.json")
print("=" * 60)


# ════════════════════════════════════════════════════════
# CÉLULA 8 — Misturar modelo histórico com a forma ATUAL da Copa
# O modelo treinado captura força histórica (150 anos de jogos),
# mas não sabe nada sobre os jogos que essas seleções fizeram
# nesta Copa específica. Esta célula corrige isso com um blend.
# ════════════════════════════════════════════════════════
from dataclasses import dataclass

@dataclass
class FormaAtual:
    nome: str
    gols_marcados: int   # nesta Copa (fase de grupos até agora)
    gols_sofridos: int
    pontos: int
    jogos: int            # quantos jogos já disputou nesta Copa

# preencha com os dados reais de cada seleção nesta Copa 2026
FORMA_ATUAL = {
    "Senegal":   FormaAtual("Senegal",   gols_marcados=3, gols_sofridos=4, pontos=2, jogos=2),
    "Iraq":      FormaAtual("Iraq",      gols_marcados=1, gols_sofridos=7, pontos=0, jogos=2),
    "Norway":    FormaAtual("Norway",    gols_marcados=7, gols_sofridos=3, pontos=6, jogos=2),
    "France":    FormaAtual("France",    gols_marcados=6, gols_sofridos=1, pontos=6, jogos=2),
    "Argentina": FormaAtual("Argentina", gols_marcados=5, gols_sofridos=0, pontos=6, jogos=2),
    "Austria":   FormaAtual("Austria",   gols_marcados=3, gols_sofridos=4, pontos=3, jogos=2),
    "Algeria":   FormaAtual("Algeria",   gols_marcados=2, gols_sofridos=4, pontos=3, jogos=2),
    "Jordan":    FormaAtual("Jordan",    gols_marcados=2, gols_sofridos=4, pontos=0, jogos=2),
}

def lambda_forma_atual(mandante_nome, visitante_nome):
    """Calcula lambda (gols esperados) baseado SÓ no desempenho desta Copa,
    usando o mesmo espírito do calcular_forca() do script original."""
    m = FORMA_ATUAL[mandante_nome]
    v = FORMA_ATUAL[visitante_nome]

    def forca(f):
        ataque   = f.gols_marcados / max(f.jogos, 1) / 3.0      # normaliza ~0-1
        saldo    = np.tanh((f.gols_marcados - f.gols_sofridos) / 5)
        aproveit = f.pontos / max(f.jogos * 3, 1)
        return max(0.4 * ataque + 0.35 * (saldo + 1) / 2 + 0.25 * aproveit, 0.05)

    forca_m, forca_v = forca(m), forca(v)
    base = 1.3
    lam_m = base * forca_m / (forca_m + forca_v) * 2.2
    lam_v = base * forca_v / (forca_m + forca_v) * 2.2

    # fator pressão: quem está pior na tabela ataca mais
    if m.pontos < v.pontos:
        lam_m *= 1.15; lam_v *= 0.90
    elif v.pontos < m.pontos:
        lam_v *= 1.15; lam_m *= 0.90

    return lam_m, lam_v


def prever_jogo_blend(mandante_nome, visitante_nome, alpha_historico=0.6, n_simulacoes=10_000):
    """
    alpha_historico: peso do modelo treinado (150 anos de histórico).
    (1 - alpha_historico): peso da forma atual nesta Copa 2026.
    alpha_historico=1.0  -> só histórico (igual prever_jogo original)
    alpha_historico=0.0  -> só forma atual desta Copa
    """
    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lam_h_modelo, lam_v_modelo = modelo(h_idx, a_idx)
        lam_h_modelo, lam_v_modelo = lam_h_modelo.item(), lam_v_modelo.item()

    lam_h_atual, lam_v_atual = lambda_forma_atual(mandante_nome, visitante_nome)

    lambda_h = alpha_historico * lam_h_modelo + (1 - alpha_historico) * lam_h_atual
    lambda_v = alpha_historico * lam_v_modelo + (1 - alpha_historico) * lam_v_atual

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_v, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_historico": (round(lam_h_modelo, 2), round(lam_v_modelo, 2)),
        "lambda_forma_atual": (round(lam_h_atual, 2), round(lam_v_atual, 2)),
        "lambda_final_blend": (round(lambda_h, 2), round(lambda_v, 2)),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# roda os 4 jogos já misturando histórico (60%) + forma atual (40%)
resultados_blend = []
for jogo in JOGOS_AMANHA:
    r = prever_jogo_blend(jogo["mandante_nome"], jogo["visitante_nome"], alpha_historico=0.6)
    r["horario"] = jogo["horario"]
    r["contexto"] = jogo["contexto"]
    r["grupo"] = jogo["grupo"]
    resultados_blend.append(r)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {r['mandante']} x {r['visitante']}")
    print(f"   λ histórico:    {r['lambda_historico']}")
    print(f"   λ forma atual:  {r['lambda_forma_atual']}")
    print(f"   λ final (blend): {r['lambda_final_blend']}")
    print(f"   📊 Vitória {r['mandante']}: {r['prob_mandante']}% | Empate: {r['prob_empate']}% | Vitória {r['visitante']}: {r['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {r['palpite_maquina']}")

with open("palpites_maquina_blend.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "blend 60% histórico (embeddings treinados) + 40% forma atual da Copa 2026",
        "jogos": resultados_blend
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados (blend) salvos em palpites_maquina_blend.json")

Usando device: cuda
GPU: NVIDIA GeForce RTX 3090
Total de partidas carregadas: 15871
Total de seleções no dataset: 312
ForcaSelecaoModel(
  (ataque): Embedding(312, 8)
  (defesa): Embedding(312, 8)
)
Época   0 | Perda: 3.7217
Época  50 | Perda: 3.3575
Época 100 | Perda: 3.0936
Época 150 | Perda: 3.0264
Época 200 | Perda: 2.9969
Época 250 | Perda: 2.9806
✅ Treinamento concluído.
  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)
  Cultura Eclética x RecomendeMe

🕐 16h (Brasília) — Grupo I
   Senegal x Iraq
   📊 Vitória Senegal: 63.2%
   📊 Empate:                  26.2%
   📊 Vitória Iraq: 10.6%
   🤖 Palpite da Máquina: 1x0
   📌 Placares mais prováveis:
      1x0 → 22.9% das simulações
      2x0 → 16.4% das simulações
      0x0 → 15.2% das simulações
   💬 Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.

🕐 16h (Brasília) — Grupo I
   Norway x France
   📊 Vitória Norway: 9.8%
   📊 Empate:                  19.9%
   📊 Vitória France: 70.2%
   🤖 Palp

In [ ]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

# remove linhas com placar ausente/inválido (causa comum de NaN no treino)
df = df.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).reset_index(drop=True)
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)].reset_index(drop=True)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        log_lambda_home = log_lambda_home.clamp(min=-5.0, max=3.0)
        log_lambda_away = log_lambda_away.clamp(min=-5.0, max=3.0)

        lambda_home = torch.exp(log_lambda_home)
        lambda_away = torch.exp(log_lambda_away)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.005)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    if torch.isnan(perda):
        print(f"⚠️ Loss virou NaN na época {epoca} — pare e revise os dados/lr.")
        break

    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)  # evita explosão de gradiente
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# ════════════════════════════════════════════════════════
# CÉLULA 7 — Rodar todos os jogos de amanhã e salvar JSON final
# Nomes no padrão do dataset martj42/international_results
# ════════════════════════════════════════════════════════
JOGOS_AMANHA = [
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Senegal",
        "visitante_nome": "Iraq",
        "contexto": "Senegal precisa vencer para brigar por vaga entre melhores 3ºs. Iraque eliminado.",
        "grupo": "I"
    },
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Norway",
        "visitante_nome": "France",
        "contexto": "Decide liderança do Grupo I. Ambas já classificadas. Jogo de prestígio.",
        "grupo": "I"
    },
    {
        "horario": "19h (Brasília)",
        "mandante_nome": "Argentina",
        "visitante_nome": "Austria",
        "contexto": "Decide quem fecha o Grupo J em 1º. Argentina já classificada, mas Áustria ainda briga por vaga.",
        "grupo": "J"
    },
    {
        "horario": "22h (Brasília)",
        "mandante_nome": "Algeria",
        "visitante_nome": "Jordan",
        "contexto": "Argélia briga pelo 2º lugar do Grupo J. Jordânia eliminada.",
        "grupo": "J"
    },
]

resultados = []

print("=" * 60)
print("  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)")
print("  Cultura Eclética x RecomendeMe")
print("=" * 60)

for jogo in JOGOS_AMANHA:
    resultado = prever_jogo(jogo["mandante_nome"], jogo["visitante_nome"])
    resultado["horario"] = jogo["horario"]
    resultado["contexto"] = jogo["contexto"]
    resultado["grupo"] = jogo["grupo"]
    resultados.append(resultado)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {resultado['mandante']} x {resultado['visitante']}")
    print(f"   📊 Vitória {resultado['mandante']}: {resultado['prob_mandante']}%")
    print(f"   📊 Empate:                  {resultado['prob_empate']}%")
    print(f"   📊 Vitória {resultado['visitante']}: {resultado['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {resultado['palpite_maquina']}")
    print(f"   📌 Placares mais prováveis:")
    for p in resultado["placares_frequentes"]:
        print(f"      {p['placar']} → {p['freq']}% das simulações")
    print(f"   💬 {jogo['contexto']}")

with open("palpites_maquina.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "embeddings ataque/defesa treinados (PyTorch + GPU)",
        "jogos": resultados
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados salvos em palpites_maquina.json")
print("=" * 60)


# ════════════════════════════════════════════════════════
# CÉLULA 8 — Misturar modelo histórico com a forma ATUAL da Copa
# O modelo treinado captura força histórica (150 anos de jogos),
# mas não sabe nada sobre os jogos que essas seleções fizeram
# nesta Copa específica. Esta célula corrige isso com um blend.
# ════════════════════════════════════════════════════════
from dataclasses import dataclass

@dataclass
class FormaAtual:
    nome: str
    gols_marcados: int   # nesta Copa (fase de grupos até agora)
    gols_sofridos: int
    pontos: int
    jogos: int            # quantos jogos já disputou nesta Copa

# preencha com os dados reais de cada seleção nesta Copa 2026
FORMA_ATUAL = {
    "Senegal":   FormaAtual("Senegal",   gols_marcados=3, gols_sofridos=4, pontos=2, jogos=2),
    "Iraq":      FormaAtual("Iraq",      gols_marcados=1, gols_sofridos=7, pontos=0, jogos=2),
    "Norway":    FormaAtual("Norway",    gols_marcados=7, gols_sofridos=3, pontos=6, jogos=2),
    "France":    FormaAtual("France",    gols_marcados=6, gols_sofridos=1, pontos=6, jogos=2),
    "Argentina": FormaAtual("Argentina", gols_marcados=5, gols_sofridos=0, pontos=6, jogos=2),
    "Austria":   FormaAtual("Austria",   gols_marcados=3, gols_sofridos=4, pontos=3, jogos=2),
    "Algeria":   FormaAtual("Algeria",   gols_marcados=2, gols_sofridos=4, pontos=3, jogos=2),
    "Jordan":    FormaAtual("Jordan",    gols_marcados=2, gols_sofridos=4, pontos=0, jogos=2),
}

def lambda_forma_atual(mandante_nome, visitante_nome):
    """Calcula lambda (gols esperados) baseado SÓ no desempenho desta Copa,
    usando o mesmo espírito do calcular_forca() do script original."""
    m = FORMA_ATUAL[mandante_nome]
    v = FORMA_ATUAL[visitante_nome]

    def forca(f):
        ataque   = f.gols_marcados / max(f.jogos, 1) / 3.0      # normaliza ~0-1
        saldo    = np.tanh((f.gols_marcados - f.gols_sofridos) / 5)
        aproveit = f.pontos / max(f.jogos * 3, 1)
        return max(0.4 * ataque + 0.35 * (saldo + 1) / 2 + 0.25 * aproveit, 0.05)

    forca_m, forca_v = forca(m), forca(v)
    base = 1.3
    lam_m = base * forca_m / (forca_m + forca_v) * 2.2
    lam_v = base * forca_v / (forca_m + forca_v) * 2.2

    # fator pressão: quem está pior na tabela ataca mais
    if m.pontos < v.pontos:
        lam_m *= 1.15; lam_v *= 0.90
    elif v.pontos < m.pontos:
        lam_v *= 1.15; lam_m *= 0.90

    return lam_m, lam_v


def prever_jogo_blend(mandante_nome, visitante_nome, alpha_historico=0.6, n_simulacoes=10_000):
    """
    alpha_historico: peso do modelo treinado (150 anos de histórico).
    (1 - alpha_historico): peso da forma atual nesta Copa 2026.
    alpha_historico=1.0  -> só histórico (igual prever_jogo original)
    alpha_historico=0.0  -> só forma atual desta Copa
    """
    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lam_h_modelo, lam_v_modelo = modelo(h_idx, a_idx)
        lam_h_modelo, lam_v_modelo = lam_h_modelo.item(), lam_v_modelo.item()

    lam_h_atual, lam_v_atual = lambda_forma_atual(mandante_nome, visitante_nome)

    lambda_h = alpha_historico * lam_h_modelo + (1 - alpha_historico) * lam_h_atual
    lambda_v = alpha_historico * lam_v_modelo + (1 - alpha_historico) * lam_v_atual

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_v, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_historico": (round(lam_h_modelo, 2), round(lam_v_modelo, 2)),
        "lambda_forma_atual": (round(lam_h_atual, 2), round(lam_v_atual, 2)),
        "lambda_final_blend": (round(lambda_h, 2), round(lambda_v, 2)),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# roda os 4 jogos já misturando histórico (60%) + forma atual (40%)
resultados_blend = []
for jogo in JOGOS_AMANHA:
    r = prever_jogo_blend(jogo["mandante_nome"], jogo["visitante_nome"], alpha_historico=0.6)
    r["horario"] = jogo["horario"]
    r["contexto"] = jogo["contexto"]
    r["grupo"] = jogo["grupo"]
    resultados_blend.append(r)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {r['mandante']} x {r['visitante']}")
    print(f"   λ histórico:    {r['lambda_historico']}")
    print(f"   λ forma atual:  {r['lambda_forma_atual']}")
    print(f"   λ final (blend): {r['lambda_final_blend']}")
    print(f"   📊 Vitória {r['mandante']}: {r['prob_mandante']}% | Empate: {r['prob_empate']}% | Vitória {r['visitante']}: {r['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {r['palpite_maquina']}")

with open("palpites_maquina_blend.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "blend 60% histórico (embeddings treinados) + 40% forma atual da Copa 2026",
        "jogos": resultados_blend
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados (blend) salvos em palpites_maquina_blend.json")

In [6]:
# ════════════════════════════════════════════════════════
# CÉLULA 1 — Setup e checagem da GPU
# ════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ════════════════════════════════════════════════════════
# CÉLULA 2 — Carregar histórico de partidas
# Baixa direto do GitHub (sempre atualizado), sem precisar do Kaggle
# Fonte: https://github.com/martj42/international_results
# ════════════════════════════════════════════════════════
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "results.csv"
)
df = pd.read_csv("results.csv")

# filtra só jogos relevantes (opcional: últimos N anos, ou só competições oficiais)
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"] >= "2010-01-01"].reset_index(drop=True)

# opcional: dar mais peso a jogos de Copa do Mundo / Eliminatórias
PESO_TORNEIO = {
    "FIFA World Cup": 3.0,
    "FIFA World Cup qualification": 1.5,
}
df["peso"] = df["tournament"].map(PESO_TORNEIO).fillna(1.0)

# remove linhas com placar ausente/inválido (causa comum de NaN no treino)
df = df.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).reset_index(drop=True)
df = df[(df["home_score"] >= 0) & (df["away_score"] >= 0)].reset_index(drop=True)

print(f"Total de partidas carregadas: {len(df)}")
df.head()


# ════════════════════════════════════════════════════════
# CÉLULA 3 — Indexar seleções
# ════════════════════════════════════════════════════════
selecoes = sorted(set(df["home_team"]) | set(df["away_team"]))
idx_selecao = {nome: i for i, nome in enumerate(selecoes)}
n_selecoes = len(selecoes)
print(f"Total de seleções no dataset: {n_selecoes}")

df["home_idx"] = df["home_team"].map(idx_selecao)
df["away_idx"] = df["away_team"].map(idx_selecao)


# ════════════════════════════════════════════════════════
# CÉLULA 4 — Modelo: embeddings de ataque/defesa por seleção
# (estilo Dixon-Coles: cada seleção tem força de ataque e defesa,
#  e os gols esperados saem de uma Poisson com esses dois fatores)
# ════════════════════════════════════════════════════════
class ForcaSelecaoModel(nn.Module):
    def __init__(self, n_selecoes, dim_embedding=8):
        super().__init__()
        self.ataque = nn.Embedding(n_selecoes, dim_embedding)
        self.defesa = nn.Embedding(n_selecoes, dim_embedding)
        self.fator_casa = nn.Parameter(torch.tensor(0.2))  # vantagem de jogar em casa
        self.bias_gols = nn.Parameter(torch.tensor(0.3))   # nível geral de gols

        nn.init.normal_(self.ataque.weight, std=0.1)
        nn.init.normal_(self.defesa.weight, std=0.1)

    def forward(self, home_idx, away_idx):
        att_home = self.ataque(home_idx)
        def_home = self.defesa(home_idx)
        att_away = self.ataque(away_idx)
        def_away = self.defesa(away_idx)

        # produto escalar ataque-vs-defesa do adversário
        log_lambda_home = (att_home * def_away).sum(-1) + self.fator_casa + self.bias_gols
        log_lambda_away = (att_away * def_home).sum(-1) + self.bias_gols

        log_lambda_home = log_lambda_home.clamp(min=-5.0, max=3.0)
        log_lambda_away = log_lambda_away.clamp(min=-5.0, max=3.0)

        lambda_home = torch.exp(log_lambda_home)
        lambda_away = torch.exp(log_lambda_away)
        return lambda_home, lambda_away

    def forca_geral(self, idx):
        """Métrica resumo: força líquida (ataque médio - defesa média), pra usar como
        substituto direto do `calcular_forca()` do script original."""
        a = self.ataque(idx).mean(-1)
        d = self.defesa(idx).mean(-1)
        return (a - d).detach()


modelo = ForcaSelecaoModel(n_selecoes, dim_embedding=8).to(device)
print(modelo)


# ════════════════════════════════════════════════════════
# CÉLULA 5 — Treinamento (Poisson NLL loss, na GPU)
# ════════════════════════════════════════════════════════
home_idx = torch.tensor(df["home_idx"].values, dtype=torch.long, device=device)
away_idx = torch.tensor(df["away_idx"].values, dtype=torch.long, device=device)
home_gols = torch.tensor(df["home_score"].values, dtype=torch.float32, device=device)
away_gols = torch.tensor(df["away_score"].values, dtype=torch.float32, device=device)
pesos = torch.tensor(df["peso"].values, dtype=torch.float32, device=device)

poisson_nll = nn.PoissonNLLLoss(log_input=False, full=True, reduction="none")
otimizador = optim.Adam(modelo.parameters(), lr=0.005)

N_EPOCAS = 300
for epoca in range(N_EPOCAS):
    otimizador.zero_grad()
    lambda_h, lambda_a = modelo(home_idx, away_idx)

    perda = (
        poisson_nll(lambda_h, home_gols) * pesos +
        poisson_nll(lambda_a, away_gols) * pesos
    ).mean()

    if torch.isnan(perda):
        print(f"⚠️ Loss virou NaN na época {epoca} — pare e revise os dados/lr.")
        break

    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)  # evita explosão de gradiente
    otimizador.step()

    if epoca % 50 == 0:
        print(f"Época {epoca:3d} | Perda: {perda.item():.4f}")

print("✅ Treinamento concluído.")


# ════════════════════════════════════════════════════════
# CÉLULA 6 — Usar o modelo treinado pra prever os jogos de amanhã
# (substitui a calcular_forca() do script original, mas agora
#  com lambda direto do modelo, sem precisar inventar pesos manuais)
# ════════════════════════════════════════════════════════
def prever_jogo(mandante_nome, visitante_nome, n_simulacoes=10_000):
    if mandante_nome not in idx_selecao or visitante_nome not in idx_selecao:
        raise ValueError(f"Seleção não encontrada no histórico: {mandante_nome} / {visitante_nome}")

    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lambda_h, lambda_a = modelo(h_idx, a_idx)
        lambda_h, lambda_a = lambda_h.item(), lambda_a.item()

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_a, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_mandante": round(lambda_h, 2),
        "lambda_visitante": round(lambda_a, 2),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# ════════════════════════════════════════════════════════
# CÉLULA 7 — Rodar todos os jogos de amanhã e salvar JSON final
# Nomes no padrão do dataset martj42/international_results
# ════════════════════════════════════════════════════════
JOGOS_AMANHA = [
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Norway",
        "visitante_nome": "France",
        "contexto": "Decide liderança do Grupo I. Ambas já classificadas. Jogo de prestígio.",
        "grupo": "I"
    },
    {
        "horario": "16h (Brasília)",
        "mandante_nome": "Senegal",
        "visitante_nome": "Iraq",
        "contexto": "Ambos eliminados na prática, brigam só por amor próprio/estatística pessoal.",
        "grupo": "I"
    },
    {
        "horario": "21h (Brasília)",
        "mandante_nome": "Cape Verde",
        "visitante_nome": "Saudi Arabia",
        "contexto": "Cabo Verde (1ª Copa da história) briga pela vaga direta. Arábia Saudita precisa vencer.",
        "grupo": "H"
    },
    {
        "horario": "21h (Brasília)",
        "mandante_nome": "Uruguay",
        "visitante_nome": "Spain",
        "contexto": "Espanha lidera e depende só de si. Uruguai briga pela vaga direta ou melhor 3º.",
        "grupo": "H"
    },
    {
        "horario": "22h (Brasília)",
        "mandante_nome": "New Zealand",
        "visitante_nome": "Belgium",
        "contexto": "Nova Zelândia precisa vencer pra sobreviver. Bélgica ainda não venceu na Copa.",
        "grupo": "G"
    },
    {
        "horario": "22h (Brasília)",
        "mandante_nome": "Egypt",
        "visitante_nome": "Iran",
        "contexto": "Egito lidera e busca confirmar 1º lugar histórico do grupo. Irã briga pela vaga direta.",
        "grupo": "G"
    },
]

resultados = []

print("=" * 60)
print("  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)")
print("  Cultura Eclética x RecomendeMe")
print("=" * 60)

for jogo in JOGOS_AMANHA:
    resultado = prever_jogo(jogo["mandante_nome"], jogo["visitante_nome"])
    resultado["horario"] = jogo["horario"]
    resultado["contexto"] = jogo["contexto"]
    resultado["grupo"] = jogo["grupo"]
    resultados.append(resultado)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {resultado['mandante']} x {resultado['visitante']}")
    print(f"   📊 Vitória {resultado['mandante']}: {resultado['prob_mandante']}%")
    print(f"   📊 Empate:                  {resultado['prob_empate']}%")
    print(f"   📊 Vitória {resultado['visitante']}: {resultado['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {resultado['palpite_maquina']}")
    print(f"   📌 Placares mais prováveis:")
    for p in resultado["placares_frequentes"]:
        print(f"      {p['placar']} → {p['freq']}% das simulações")
    print(f"   💬 {jogo['contexto']}")

with open("palpites_maquina.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "embeddings ataque/defesa treinados (PyTorch + GPU)",
        "jogos": resultados
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados salvos em palpites_maquina.json")
print("=" * 60)


# ════════════════════════════════════════════════════════
# CÉLULA 8 — Misturar modelo histórico com a forma ATUAL da Copa
# O modelo treinado captura força histórica (150 anos de jogos),
# mas não sabe nada sobre os jogos que essas seleções fizeram
# nesta Copa específica. Esta célula corrige isso com um blend.
# ════════════════════════════════════════════════════════
from dataclasses import dataclass

@dataclass
class FormaAtual:
    nome: str
    gols_marcados: int   # nesta Copa (fase de grupos até agora)
    gols_sofridos: int
    pontos: int
    jogos: int            # quantos jogos já disputou nesta Copa

# preencha com os dados reais de cada seleção nesta Copa 2026
FORMA_ATUAL = {
    "Senegal":      FormaAtual("Senegal",      gols_marcados=3, gols_sofridos=6, pontos=0, jogos=2),
    "Iraq":         FormaAtual("Iraq",         gols_marcados=1, gols_sofridos=7, pontos=0, jogos=2),
    "Norway":       FormaAtual("Norway",       gols_marcados=7, gols_sofridos=3, pontos=6, jogos=2),
    "France":       FormaAtual("France",       gols_marcados=6, gols_sofridos=1, pontos=6, jogos=2),
    "Cape Verde":   FormaAtual("Cape Verde",   gols_marcados=2, gols_sofridos=2, pontos=2, jogos=2),
    "Saudi Arabia": FormaAtual("Saudi Arabia", gols_marcados=1, gols_sofridos=5, pontos=1, jogos=2),
    "Uruguay":      FormaAtual("Uruguay",      gols_marcados=3, gols_sofridos=3, pontos=2, jogos=2),
    "Spain":        FormaAtual("Spain",        gols_marcados=4, gols_sofridos=0, pontos=4, jogos=2),
    "New Zealand":  FormaAtual("New Zealand",  gols_marcados=3, gols_sofridos=5, pontos=1, jogos=2),
    "Belgium":      FormaAtual("Belgium",      gols_marcados=1, gols_sofridos=1, pontos=2, jogos=2),
    "Egypt":        FormaAtual("Egypt",        gols_marcados=4, gols_sofridos=2, pontos=4, jogos=2),
    "Iran":         FormaAtual("Iran",         gols_marcados=2, gols_sofridos=2, pontos=2, jogos=2),
}

def lambda_forma_atual(mandante_nome, visitante_nome):
    """Calcula lambda (gols esperados) baseado SÓ no desempenho desta Copa,
    usando o mesmo espírito do calcular_forca() do script original."""
    m = FORMA_ATUAL[mandante_nome]
    v = FORMA_ATUAL[visitante_nome]

    def forca(f):
        ataque   = f.gols_marcados / max(f.jogos, 1) / 3.0      # normaliza ~0-1
        saldo    = np.tanh((f.gols_marcados - f.gols_sofridos) / 5)
        aproveit = f.pontos / max(f.jogos * 3, 1)
        return max(0.4 * ataque + 0.35 * (saldo + 1) / 2 + 0.25 * aproveit, 0.05)

    forca_m, forca_v = forca(m), forca(v)
    base = 1.3
    lam_m = base * forca_m / (forca_m + forca_v) * 2.2
    lam_v = base * forca_v / (forca_m + forca_v) * 2.2

    # fator pressão: quem está pior na tabela ataca mais
    if m.pontos < v.pontos:
        lam_m *= 1.15; lam_v *= 0.90
    elif v.pontos < m.pontos:
        lam_v *= 1.15; lam_m *= 0.90

    return lam_m, lam_v


def prever_jogo_blend(mandante_nome, visitante_nome, alpha_historico=0.6, n_simulacoes=10_000):
    """
    alpha_historico: peso do modelo treinado (150 anos de histórico).
    (1 - alpha_historico): peso da forma atual nesta Copa 2026.
    alpha_historico=1.0  -> só histórico (igual prever_jogo original)
    alpha_historico=0.0  -> só forma atual desta Copa
    """
    h_idx = torch.tensor([idx_selecao[mandante_nome]], device=device)
    a_idx = torch.tensor([idx_selecao[visitante_nome]], device=device)

    with torch.no_grad():
        lam_h_modelo, lam_v_modelo = modelo(h_idx, a_idx)
        lam_h_modelo, lam_v_modelo = lam_h_modelo.item(), lam_v_modelo.item()

    lam_h_atual, lam_v_atual = lambda_forma_atual(mandante_nome, visitante_nome)

    lambda_h = alpha_historico * lam_h_modelo + (1 - alpha_historico) * lam_h_atual
    lambda_v = alpha_historico * lam_v_modelo + (1 - alpha_historico) * lam_v_atual

    gols_m = np.random.poisson(lambda_h, n_simulacoes)
    gols_v = np.random.poisson(lambda_v, n_simulacoes)

    prob_m = float((gols_m > gols_v).mean() * 100)
    prob_e = float((gols_m == gols_v).mean() * 100)
    prob_v = float((gols_m < gols_v).mean() * 100)

    from collections import Counter
    placares = Counter(zip(gols_m, gols_v)).most_common(3)

    return {
        "mandante": mandante_nome,
        "visitante": visitante_nome,
        "lambda_historico": (round(lam_h_modelo, 2), round(lam_v_modelo, 2)),
        "lambda_forma_atual": (round(lam_h_atual, 2), round(lam_v_atual, 2)),
        "lambda_final_blend": (round(lambda_h, 2), round(lambda_v, 2)),
        "prob_mandante": round(prob_m, 1),
        "prob_empate": round(prob_e, 1),
        "prob_visitante": round(prob_v, 1),
        "placares_frequentes": [
            {"placar": f"{int(p[0][0])}x{int(p[0][1])}", "freq": round(float(p[1])/n_simulacoes*100, 1)}
            for p in placares
        ],
        "palpite_maquina": f"{int(placares[0][0][0])}x{int(placares[0][0][1])}",
    }


# roda os 4 jogos já misturando histórico (60%) + forma atual (40%)
resultados_blend = []
for jogo in JOGOS_AMANHA:
    r = prever_jogo_blend(jogo["mandante_nome"], jogo["visitante_nome"], alpha_historico=0.6)
    r["horario"] = jogo["horario"]
    r["contexto"] = jogo["contexto"]
    r["grupo"] = jogo["grupo"]
    resultados_blend.append(r)

    print(f"\n🕐 {jogo['horario']} — Grupo {jogo['grupo']}")
    print(f"   {r['mandante']} x {r['visitante']}")
    print(f"   λ histórico:    {r['lambda_historico']}")
    print(f"   λ forma atual:  {r['lambda_forma_atual']}")
    print(f"   λ final (blend): {r['lambda_final_blend']}")
    print(f"   📊 Vitória {r['mandante']}: {r['prob_mandante']}% | Empate: {r['prob_empate']}% | Vitória {r['visitante']}: {r['prob_visitante']}%")
    print(f"   🤖 Palpite da Máquina: {r['palpite_maquina']}")

with open("palpites_maquina_blend.json", "w", encoding="utf-8") as f:
    json.dump({
        "rodada": "3ª rodada — Fase de Grupos",
        "data": "26/06/2026",
        "modelo": "blend 60% histórico (embeddings treinados) + 40% forma atual da Copa 2026",
        "jogos": resultados_blend
    }, f, ensure_ascii=False, indent=2)

print("\n✅ Resultados (blend) salvos em palpites_maquina_blend.json")

Usando device: cuda
GPU: NVIDIA GeForce RTX 3090
Total de partidas carregadas: 15871
Total de seleções no dataset: 312
ForcaSelecaoModel(
  (ataque): Embedding(312, 8)
  (defesa): Embedding(312, 8)
)
Época   0 | Perda: 3.7213
Época  50 | Perda: 3.3669
Época 100 | Perda: 3.0949
Época 150 | Perda: 3.0285
Época 200 | Perda: 2.9991
Época 250 | Perda: 2.9827
✅ Treinamento concluído.
  PALPITES DA MÁQUINA — Copa do Mundo 2026 (modelo treinado)
  Cultura Eclética x RecomendeMe

🕐 16h (Brasília) — Grupo I
   Norway x France
   📊 Vitória Norway: 26.5%
   📊 Empate:                  21.5%
   📊 Vitória France: 52.0%
   🤖 Palpite da Máquina: 1x2
   📌 Placares mais prováveis:
      1x2 → 9.3% das simulações
      1x1 → 8.9% das simulações
      2x2 → 6.8% das simulações
   💬 Decide liderança do Grupo I. Ambas já classificadas. Jogo de prestígio.

🕐 16h (Brasília) — Grupo I
   Senegal x Iraq
   📊 Vitória Senegal: 78.1%
   📊 Empate:                  15.4%
   📊 Vitória Iraq: 6.5%
   🤖 Palpite da Máquin